# Actividad 2026 · Unidad 4

**Atribución:** esta actividad adapta material del [Deep Reinforcement Learning Course de Hugging Face](https://huggingface.co/learn/deep-rl-course). El contenido original pertenece a sus autores y colaboradores.

**Traducción:** texto en español elaborado con asistencia de IA a partir del material original. Puede contener errores; ante una duda, consultá el notebook fuente.

[Abrir esta unidad en Colab](https://colab.research.google.com/drive/1Gn-X8RCJTNW4F8QcRAPfopjf9t0t8tOJ)

Adaptación del [notebook oficial](https://github.com/huggingface/deep-rl-class/blob/884eddd749ffbf2d0571082f02c18330266b3d6e/notebooks/unit4/unit4.ipynb). Implementa REINFORCE con PyTorch en CartPole-v1 usando Gymnasium. Las extensiones de PixelCopter y publicación se excluyeron.


In [ ]:
%pip install -q gymnasium matplotlib


In [ ]:
import numpy as np

from collections import deque

import matplotlib.pyplot as plt
%matplotlib inline

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

# Gym
import gymnasium as gym

# Hugging Face Hub

import imageio

## Comprobá si hay una GPU

La siguiente celda muestra el dispositivo disponible. Si Colab asignó una GPU, debería aparecer `device:cuda:0`.


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
print(device)

Ya podemos implementar REINFORCE 🔥

# Primer agente: CartPole-v1 🤖

## Creá y explorá el entorno CartPole

### [Documentación del entorno 🎮](https://gymnasium.farama.org/environments/classic_control/cart_pole/)


### ¿Por qué empezar con CartPole-v1?

Al implementar un agente desde cero, conviene detectar errores primero en un entorno sencillo. Los [consejos de Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html) recomiendan verificar señales de aprendizaje en problemas pequeños antes de pasar a otros más complejos.

### El entorno CartPole-v1

Un poste está unido a un carrito que se desplaza sobre una pista. El objetivo es mover el carrito a izquierda o derecha para **mantener el poste equilibrado**.

El episodio termina cuando la inclinación supera ±12°, el carrito se aleja más de ±2,4 unidades del centro o se alcanza el límite de 500 pasos. El agente recibe **+1 por cada paso** en que mantiene el equilibrio.


In [ ]:
env_id = "CartPole-v1"
# Crear el entorno
env = gym.make(env_id)

# Crear el entorno de evaluación
eval_env = gym.make(env_id)

# Obtener los espacios de estados y acciones
s_size = env.observation_space.shape[0]
a_size = env.action_space.n

In [ ]:
print("_____ESPACIO DE OBSERVACIÓN_____ \n")
print("Dimensión del estado: ", s_size)
print("Observación de ejemplo", env.observation_space.sample()) # Obtener una observación aleatoria

In [ ]:
print("\n _____ESPACIO DE ACCIONES_____ \n")
print("Cantidad de acciones: ", a_size)
print("Acción de ejemplo", env.action_space.sample()) # Elegir una acción al azar

## Construí la arquitectura de REINFORCE

Esta actividad se basa en el [ejemplo oficial de PyTorch](https://github.com/pytorch/examples/blob/main/reinforcement_learning/reinforce.py), el [notebook de Udacity](https://github.com/udacity/deep-reinforcement-learning/blob/master/reinforce/REINFORCE.ipynb) y una [mejora aportada por Chris1nexus](https://github.com/huggingface/deep-rl-class/pull/95).

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit6/reinforce.png" alt="Arquitectura REINFORCE"/>


La red tendrá dos capas totalmente conectadas (`fc1` y `fc2`), activación ReLU después de `fc1` y Softmax a la salida para obtener una distribución de probabilidad sobre las acciones.


**Ejercicio: completar antes de ejecutar.** La solución aparece más adelante cuando está incluida en el original.

```python
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        # Crear dos capas totalmente conectadas



    def forward(self, x):
        # Definir la pasada hacia adelante
        # El estado pasa por fc1 y luego por ReLU

        # La salida de fc1 pasa por fc2

        # La salida se transforma con Softmax
    
    def act(self, state):
        """
        Dado un estado, elegir una acción
        """
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = np.argmax(m)
        return action.item(), m.log_prob(action)
```


### Solución

In [ ]:
# @title Solución (Mostrar código) { display-mode: "form" }
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, a_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.softmax(x, dim=1)
    
    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = np.argmax(m)
        return action.item(), m.log_prob(action)

Hay un error en esta implementación. ¿Podés encontrarlo? Probemos una pasada hacia adelante.


**Extensión del notebook original; se omite en la actividad 2026.**


Aparece `ValueError: The value argument to log_prob must be a Tensor`: `action` no es un tensor, aunque `m.log_prob(action)` lo requiere. Revisá `act` y recordá que queremos **muestrear una acción** de la distribución de probabilidades.


### Solución corregida

In [ ]:
# @title Solución (Mostrar código) { display-mode: "form" }
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, a_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.softmax(x, dim=1)
    
    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)

CartPole facilita la depuración: si algo falla en este entorno simple, podemos revisar primero nuestra implementación.


No debemos elegir siempre `action = np.argmax(m)`: eso selecciona la acción más probable y elimina el muestreo. Usamos `action = m.sample()` para tomar una acción según la distribución `P(·|s)`.


## Construí el algoritmo de entrenamiento REINFORCE

Este es el pseudocódigo:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit6/pg_pseudocode.png" alt="Pseudocódigo de gradiente de política"/>


El retorno `G_t` suma recompensas descontadas **desde el instante `t`**. Las recompensas anteriores no pueden ser consecuencia de la acción actual; por eso se usa el retorno futuro (*reward to go*). La explicación [«Don't let the past distract you»](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html#don-t-let-the-past-distract-you) desarrolla esta idea.

Para calcular eficientemente el retorno de cada paso, recorremos las recompensas en sentido inverso y reutilizamos el retorno del paso siguiente. La [contribución original](https://github.com/huggingface/deep-rl-class/pull/95) explica el procedimiento.


¿Por qué **minimizamos una pérdida** si buscamos maximizar el retorno? PyTorch optimiza por descenso de gradiente. Definimos una pérdida cuyo mínimo corresponde a una mayor probabilidad para las acciones que obtuvieron buenos retornos. En el código se usa el negativo de `log_prob` multiplicado por el retorno; al minimizarlo, aumentamos la probabilidad de las acciones favorecidas por la señal de aprendizaje.


**Ejercicio: completar antes de ejecutar.** La solución aparece más adelante cuando está incluida en el original.

```python
def reinforce(policy, optimizer, n_training_episodes, max_t, gamma, print_every):
    # Guardar la recompensa durante el entrenamiento
    scores_deque = deque(maxlen=100)
    scores = []
    # Línea 3 del pseudocódigo
    for i_episode in range(1, n_training_episodes+1):
        saved_log_probs = []
        rewards = []
        state = # TODO: reiniciar el entorno
        # Línea 4 del pseudocódigo
        for t in range(max_t):
            action, log_prob = # TODO: elegir la acción
            saved_log_probs.append(log_prob)
            state, reward, done, _ = # TODO: avanzar un paso en el entorno
            rewards.append(reward)
            if done:
                break 
        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))
        
        # Línea 6: calcular el retorno
        returns = deque(maxlen=max_t) 
        n_steps = len(rewards) 
        # Calcular los retornos descontados de cada paso,
        # sumando la recompensa actual y el retorno futuro descontado
        
        # En O(N), donde N es la cantidad de pasos
        # Esta definición de G_t sigue la de Sutton y Barto, 
        # página 44 del borrador de la segunda edición (2017).
        # G_t = r_(t+1) + r_(t+2) + ...
        
        # Calculamos cada retorno a partir del retorno futuro: 
        # reutilizamos G_(t+1) para obtener G_t.
        # G_t = r_(t+1) + gamma*G_(t+1)
        # G_(t-1) = r_t + gamma* G_t
        # Esto es programación dinámica: reutilizamos resultados 
        # para evitar cálculos repetidos.
        
        # La expansión equivalente se muestra en Sutton y Barto, página 46.
        # G_(t-1) = r_t + gamma*r_(t+1) + gamma*gamma*r_(t+2) + ...
        
        
        ## Calculamos cada retorno como: 
        #               gamma[t] * return[t] + reward[t]
        #
        ## Recorremos los pasos del último al primero para
        ## aplicar la fórmula y evitar cálculos redundantes 
        ## que aparecerían en el recorrido directo.
        
        ## La cola "returns" mantiene los retornos en orden cronológico.
        ## appendleft() inserta al inicio en tiempo constante O(1);
        ## una lista de Python necesitaría O(N).
        for t in range(n_steps)[::-1]:
            disc_return_t = (returns[0] if len(returns)>0 else 0)
            returns.appendleft(    ) # TODO: completar aquí        
       
        ## Estandarizar los retornos estabiliza el entrenamiento
        eps = np.finfo(np.float32).eps.item()
        
        ## eps es el menor flotante representable; 
        # se suma a la desviación estándar para evitar inestabilidad numérica
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)
        
        # Línea 7:
        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
        policy_loss = torch.cat(policy_loss).sum()
        
        # Línea 8: PyTorch usa descenso de gradiente 
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()
        
        if i_episode % print_every == 0:
            print('Episodio {}\tRecompensa media: {:.2f}'.format(i_episode, np.mean(scores_deque)))
        
    return scores
```


#### Solución

In [ ]:
# @title Solución (Mostrar código) { display-mode: "form" }
def reinforce(policy, optimizer, n_training_episodes, max_t, gamma, print_every):
    # Guardar la recompensa durante el entrenamiento
    scores_deque = deque(maxlen=100)
    scores = []
    # Línea 3 del pseudocódigo
    for i_episode in range(1, n_training_episodes+1):
        saved_log_probs = []
        rewards = []
        state, _ = env.reset()
        # Línea 4 del pseudocódigo
        for t in range(max_t):
            action, log_prob = policy.act(state)
            saved_log_probs.append(log_prob)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            rewards.append(reward)
            if done:
                break 
        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))
        
        # Línea 6: calcular el retorno
        returns = deque(maxlen=max_t) 
        n_steps = len(rewards) 
        # Calcular los retornos descontados de cada paso,
        # as 
        #      sumando la recompensa actual y el retorno futuro descontado
        #
        # En O(N), donde N es la cantidad de pasos
        # Esta definición de G_t sigue la de Sutton y Barto, 
        # página 44 del borrador de la segunda edición (2017).
        # G_t = r_(t+1) + r_(t+2) + ...
        
        # Calculamos cada retorno a partir del retorno futuro: 
        # reutilizamos G_(t+1) para obtener G_t.
        # G_t = r_(t+1) + gamma*G_(t+1)
        # G_(t-1) = r_t + gamma* G_t
        # Esto es programación dinámica: reutilizamos resultados 
        # para evitar cálculos repetidos.
        
        # La expansión equivalente se muestra en Sutton y Barto, página 46.
        # G_(t-1) = r_t + gamma*r_(t+1) + gamma*gamma*r_(t+2) + ...
        
        
        ## Calculamos cada retorno como: 
        #               gamma[t] * return[t] + reward[t]
        #
        ## Recorremos los pasos del último al primero para
        ## aplicar la fórmula y evitar cálculos redundantes 
        ## que aparecerían en el recorrido directo.
        
        ## La cola "returns" mantiene los retornos en orden cronológico.
        ## appendleft() inserta al inicio en tiempo constante O(1);
        ## una lista de Python necesitaría O(N).
        for t in range(n_steps)[::-1]:
            disc_return_t = (returns[0] if len(returns)>0 else 0)
            returns.appendleft( gamma*disc_return_t + rewards[t]   )    
            
        ## Estandarizar los retornos estabiliza el entrenamiento
        eps = np.finfo(np.float32).eps.item()
        ## eps es el menor flotante representable; 
        # se suma a la desviación estándar para evitar inestabilidad numérica        
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)
        
        # Línea 7:
        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
        policy_loss = torch.stack(policy_loss).sum()
        
        # Línea 8: PyTorch usa descenso de gradiente 
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()
        
        if i_episode % print_every == 0:
            print('Episodio {}\tRecompensa media: {:.2f}'.format(i_episode, np.mean(scores_deque)))
        
    return scores

## Entrená el agente

Ya podemos entrenarlo. La siguiente celda define los hiperparámetros: podés modificarlos y observar cómo cambia el resultado.


In [ ]:
cartpole_hyperparameters = {
    "h_size": 16,
    "n_training_episodes": 1000,
    "n_evaluation_episodes": 10,
    "max_t": 1000,
    "gamma": 1.0,
    "lr": 1e-2,
    "env_id": env_id,
    "state_space": s_size,
    "action_space": a_size,
}

In [ ]:
# Crear la política y enviarla al dispositivo
cartpole_policy = Policy(cartpole_hyperparameters["state_space"], cartpole_hyperparameters["action_space"], cartpole_hyperparameters["h_size"]).to(device)
cartpole_optimizer = optim.Adam(cartpole_policy.parameters(), lr=cartpole_hyperparameters["lr"])

In [ ]:
scores = reinforce(cartpole_policy,
                   cartpole_optimizer,
                   cartpole_hyperparameters["n_training_episodes"], 
                   cartpole_hyperparameters["max_t"],
                   cartpole_hyperparameters["gamma"], 
                   100)

## Definí el método de evaluación 📝

Usaremos este método para probar el agente REINFORCE después del entrenamiento.


In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, policy):
  """
  Evaluate the agent for ``n_eval_episodes`` episodes and returns average reward and std of reward.
  :param env: The evaluation environment
  :param n_eval_episodes: Number of episode to evaluate the agent
  :param policy: The Reinforce agent
  """
  episode_rewards = []
  for episode in range(n_eval_episodes):
    state, _ = env.reset()
    step = 0
    done = False
    total_rewards_ep = 0
    
    for step in range(max_steps):
      action, _ = policy.act(state)
      new_state, reward, terminated, truncated, info = env.step(action)
      done = terminated or truncated
      total_rewards_ep += reward
        
      if done:
        break
      state = new_state
    episode_rewards.append(total_rewards_ep)
  mean_reward = np.mean(episode_rewards)
  std_reward = np.std(episode_rewards)

  return mean_reward, std_reward

## Evaluá el agente 📈

In [ ]:
evaluate_agent(eval_env, 
               cartpole_hyperparameters["max_t"], 
               cartpole_hyperparameters["n_evaluation_episodes"],
               cartpole_policy)

## Para discutir
Compará la evolución de las recompensas durante el entrenamiento con el promedio final de evaluación. ¿Qué aporta la normalización de retornos en REINFORCE?
